## Newtons Gravitationsgesetz
Die Simulation soll mit Newtons Gravitationsgesetz implementiert werden:

$$
\vec{F}_{ji} = -G \cdot \frac{m_i \cdot m_j}{\|\vec{r}_{ji}\|^2}
$$

- $F_{ji}$ ist dabei die Kraft, die ein Körper $j$ auf einen Körper $i$ auswirkt.
- $m_i, m_j$ sind die Massen der beiden Körper und
- $\vec{r}_{ji}$ ist deren Abstand.
- $G$ ist die Gravitationskonstante mit $G\approx 6,67430\cdot 10^{-11}\frac{m^3}{kg\cdot s^2}$.
- Die wirkende Kraft ist negativ, da es sich um eine anziehende Kraft handelt.
- $\vec{F}_{ji} = -\vec{F}_{ij}$ da beide Körper eine betragsmäßig gleichgroße Kraft in entgegengesetzte Richtung aufeinander wirken (Wechselwirkungsgesetz).

Mit diesem Gravitationsgesetz soll nun die Simulation implementiert werden.

In [13]:
import numpy as np

class Body:
    pass

class Simulation:
    def __init__(self, time_step):
        self.bodies = []
        self.G = 6.67430e-11
        self.time_step = time_step
        self.history = {}

    def add_body(self, body):
        if any(b.name == body.name for b in self.bodies):
            raise ValueError(f"Ein Himmelskörper mit dem Namen '{body.name}' existiert bereits!")
        self.bodies.append(body)
        self.history[body.name] = []

    def remove_body(self, body):
        if body in self.bodies:
            self.bodies.remove(body)

    def calculate_gravity(self):
        # Dictionary mit allen Körpern
        forces = {body.name : np.zeros(3) for body in self.bodies}

        for i in range(len(self.bodies)):
            for j in range(i + 1, len(self.bodies)):
                b1 = self.bodies[i]
                b2 = self.bodies[j]
                
                # Abstandsvektor von b2 zu b1
                r_vector = b1.position - b2.position
                distance = np.linalg.norm(r_vector)
                
                if distance == 0:
                    continue # zur Sicherheit gegen Division durch Null
                
                # Berechne den Betrag der Kraft nach Newton
                force_magnitude = self.G * b1.mass * b2.mass / (distance**2)
                
                # Richtungsvektor (Einheitsvektor)
                direction = r_vector / distance
                
                # Gesamtkraftvektor
                force_vector = force_magnitude * direction
                
                # Wechselwirkungsgesetz
                forces[b2.name] += force_vector  # b2 wird von b1 angezogen
                forces[b1.name] -= force_vector  # b1 wird von b2 angezogen
                
        return forces

    def check_collisions(self):
        pass

    def merge_bodies(self, body1, body2):
        pass

    def step(self):
        self.check_collisions()
        
        # Kräfte berechnen
        forces = self.calculate_gravity()
        
        for body in self.bodies:
            if body.name in forces: # zur Sicherheit gegen Fehler
                body.update_velocity(forces[body.name], self.time_step)
                body.update_position(self.time_step)
    
                # Verlauf speichern für die spätere Visualisierung
                self.history[body.name].append(body.position.copy())

    def run(self, total_time):
        # Berechnet, wie viele Schritte nötig sind
        steps = int(total_time / self.time_step)
        for _ in range(steps):
            self.step()

simulation = Simulation(20)

Die Klasse Simulation implementiert die Physik des Programms. Sie agiert als diskrete Physik-Engine, welche die zeitliche Evolution des Mehrkörper-Systems (N-Körper-Problem) auf Basis der klassischen Newtonschen Mechanik berechnet.  

Die Klasse erfüllt dabei folgende Kernaufgaben:
- Verwaltung des Systems: Sie registriert alle am System beteiligten Himmelskörper (Body-Objekte) und initialisiert Datenstrukturen für die Speicherung der Trajektorien.

- Numerische Integration: Da eine analytische Lösung für N>2 Körper nicht existiert, nutzt die Engine ein numerisches Integrationsverfahren. Sie berechnet in jedem konfigurierbaren Zeitschritt ($\Delta t$) die aktuellen Beschleunigungen, Geschwindigkeiten und Positionen aller Körper.

- Gravitative Wechselwirkung: Über die Methode calculate_gravity wird paarweise der Gravitationsvektor nach Newtons Gravitationsgesetz ermittelt.  

- Kollisionsmanagement: In jedem Simulationsschritt wird geprüft, ob sich die physikalischen Radien zweier Körper überschneiden. Bei Kontakt wird eine 100% inelastische Kollision simuliert, bei der die Massen und Impulse addiert werden und ein neuer, volumenbasierter Radius berechnet wird.

- Historisierung: Für die anschließende Visualisierung trackt die Klasse alle vergangenen Positionen (history), sodass Flugbahnen und Animationen exakt rekonstruiert werden können.